# 2 · Compare and decide

**What would make the repair acceptable?** Give the app the current policy document. Keep the test cases and checks the same.


In [ ]:
import importlib
import os, sys, json
from pathlib import Path

# Find the repository folder in local Jupyter or Databricks.
start = Path(os.environ.get("WORKSHOP_PROJECT_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "west_workshop" / "data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set WORKSHOP_PROJECT_ROOT to the workshop repository folder, then restart this notebook.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
os.chdir(ROOT)
MODE = os.environ.get("WORKSHOP_MODE", "saved")  # Choose live mode to make new model calls.


In [ ]:
from IPython.display import display, HTML
from west_workshop.repair_workshop import notebook_session
from west_workshop.mlflow_view import (
    session_header, controls_card, policy_card, inspection_card,
    finding_cases, paired_summary, review_card, decision_card, statistics_card,
)

session = notebook_session(MODE)
display(HTML(session_header(session)))


## Test the judge

The judge is a model that grades answers. Controls are examples with known grades that we use to test it.

Can it distinguish a customer's request from an assistant's claim that it processed a refund?


In [ ]:
controls = None
if not session.has_execution_failure():
    controls = session.qualify_judge()
display(HTML(controls_card(controls)))


## Evaluate the repair

Use the same MLflow call for both versions. Stop further model calls if the judge fails the controls or execution fails.


In [ ]:
if MODE == "live" and controls and controls["passed"] and not session.has_execution_failure():
    import mlflow
    for variant in ("deployed", "repair"):
        if session.needs(variant):
            with session.evaluation(variant) as run:
                result = mlflow.genai.evaluate(
                    data=run.data, predict_fn=run.predict_fn, scorers=run.scorers,
                )
                session.record_evaluation(variant, result, run)
            if session.has_execution_failure():
                break

review_path = os.environ.get("WORKSHOP_REVIEW_JSON")
reviewed = json.loads(Path(review_path).read_text(encoding="utf-8")) if review_path else None
comparison = session.policy_inputs(reviewed)


## Code view 3: apply the acceptance rules

MLflow records the results. These rules decide whether the repair passes.


In [ ]:
from west_workshop.repair_policy import DEFAULT_POLICY, evaluate_repair

policy_result = evaluate_repair(**comparison, policy=DEFAULT_POLICY)


In [ ]:
report = session.finish(comparison["review"])
assert report["policy_result"] == policy_result
display(HTML(policy_card(DEFAULT_POLICY)))


## Inspect in MLflow: repaired answer → review findings

Open `day_45_opening` for the repair. Check the policy and answer, then inspect any case flagged in the review.
MLflow records what happened and the scores. The review records any issue that still needs attention.


In [ ]:
display(HTML(inspection_card(session, "repair", ["day_45_opening", *finding_cases(report)])))


## Compare each case and read the review

The table shows each case's overall result. For each protected check, 1 to 0 is a new failure, even if the overall result stays 0. A check that stays at 0 is a tie.
A completed review can still have an open issue that affects the decision.


In [ ]:
display(HTML(paired_summary(report)))


In [ ]:
display(HTML(review_card(report)))


## Vote before revealing the decision

**Pause for 20 seconds.** Which decision fits the evidence?

- **PASS**: all acceptance rules met.
- **BLOCK**: an acceptance rule failed.
- **REVIEW REQUIRED**: review unfinished or an important issue still open.
- **INVALID**: results cannot be compared.

Name the evidence that would change your decision.


In [ ]:
display(HTML(decision_card(report)))


## Turn the finding into a future test

Keep the original answer and score. Ask a person to approve the expected grade for the new test case.
Test the updated judge on controls, including new examples it was not tuned on. Then use it to grade all the same saved answers again. Keep both sets of scores.

The README shows where to change the app, cases, checks and acceptance rules. Write down who makes the next decision and what evidence they need.


In [ ]:
display(HTML(statistics_card(report)))
